# Tutorial 1 — Biological sequence landscape (`ProteinLandscape`)

**Dataset:** Mira *et al.* 2015 — TEM β-lactamase under cefotaxime (CTX) selection.

Combinatorially complete 4-position protein library (M/L, E/K, G/S, N/D — 2⁴ = 16 variants). Fitness `f` is log growth in CTX, a classical antibiotic-resistance landscape.

This notebook shows the **sequence-landscape** workflow: pass strings (or per-position columns) to `ProteinLandscape`, which selects the protein alphabet and the Hamming-1 neighbourhood automatically.

In [ ]:
import warnings
import pandas as pd
from graphfla.landscape import ProteinLandscape
from graphfla.analysis import (
    lo_ratio, fitness_distance_corr, autocorrelation,
    classify_epistasis, global_optima_accessibility,
)

warnings.filterwarnings('ignore', category=UserWarning)  # alphabet-coverage warning

df = pd.read_csv('../data/BioSequence/Mira2015_TEM_CTX.csv')
df.head()

## 1. Build the landscape

The sequence handler accepts either a list of strings (`'MEGN'`, `'MEGD'`, ...) or a DataFrame whose columns are positions. We pass `pos1..pos4` and the protein alphabet is inferred internally.

In [ ]:
X = df[['pos1', 'pos2', 'pos3', 'pos4']]
f = df['fitness']

ls = ProteinLandscape(maximize=True)
ls.build_from_data(X, f, verbose=False)

print(f'nodes: {ls.graph.vcount()}, edges: {ls.graph.ecount()}')

## 2. Topographical features

A small panel covering the four GraphFLA dimensions: ruggedness (`lo_ratio`, `autocorrelation`), navigability (`fitness_distance_corr`, `global_optima_accessibility`), and epistasis (`classify_epistasis`).

In [ ]:
print(f'Local-optima ratio       : {lo_ratio(ls):.4f}')
print(f'Autocorrelation (lag 1)  : {autocorrelation(ls):.4f}')
print(f'Fitness-distance corr    : {fitness_distance_corr(ls):.4f}')
print(f'Global-optimum access    : {global_optima_accessibility(ls):.4f}')

epi = classify_epistasis(ls)
print('\nEpistasis classification (fraction of 4-node squares):')
for k, v in epi.items():
    print(f'  {k:>26}: {v:.4f}')

## 3. One plot — fitness vs. distance to the global optimum

In [ ]:
from graphfla.plotting import draw_fitness_distance_corr
draw_fitness_distance_corr(ls)

## Notes
- Use `DNALandscape` / `RNALandscape` exactly the same way for nucleotide alphabets.
- For arbitrary sequence alphabets use `SequenceLandscape(alphabet=[...])`.
- The protein-alphabet warning (only 8/20 characters present) is expected for binary-site libraries like this one. It does not affect the topology — only the alphabet coverage check.